# Lab 1 — The dataset contract: chip it, split it, baseline it, document it
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trongan93/dl-space-2026/blob/main/notebooks/Week3_Lab1_Dataset_Contract.ipynb)  ·  repo: [trongan93/dl-space-2026](https://github.com/trongan93/dl-space-2026)

**Deep Learning in Space Technology Applications · 115-1 · Week 3 (21 Sep 2026) · due Sunday 27 Sep 2026, 23:59 on i-School Plus**

| | |
|---|---|
| Name / student ID (or team) | *fill in* |
| Source cube | `lab0_cube.npy`, `lab0_scl.npy`, `lab0_meta.json` from Lab 0 (Taipei), **or** the team's seed area |

### What you will do
1. Turn the Scene Classification Layer into a **weak water label** with an ignore value, and hand-check ten chips.
2. **Chip** the cube into 128 × 128 tensors with bookkeeping (row, col, tile, date, valid fraction).
3. Build **two splits** of the same chips: random, and spatial blocks. Measure the leakage gap.
4. Score two **baselines** honestly: Otsu on NDWI, and a small random forest on bands + indices + texture.
5. Turn the NDWI mask into **objects** (connected components) and count them.
6. Fill the **dataset card** (separate template) with the numbers this notebook prints.

Runs on Colab in ~10 minutes (the random forest is the slow part). Upload the three Lab 0 files to the session first, or set `USE_SYNTHETIC = True` to test the pipeline on a synthetic scene (not valid for submission).

In [ ]:
import importlib, subprocess, sys
for pkg, mod in [("scikit-learn", "sklearn"), ("scikit-image", "skimage")]:
    try: importlib.import_module(mod)
    except ImportError: subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)
import os, json, collections, numpy as np, pandas as pd, matplotlib.pyplot as plt
from skimage.filters import threshold_otsu
from skimage import measure
from scipy import ndimage
USE_SAMPLE = False
if not os.path.exists("lab0_cube.npy"):
    # no Lab 0 files uploaded: try the course's bundled real Sentinel-2 sample before giving up
    try:
        import urllib.request, rasterio
        for fn in ("taipei_s2_sample.tif", "taipei_s2_sample_scl.tif"):
            if not os.path.exists(fn): urllib.request.urlretrieve("https://raw.githubusercontent.com/trongan93/dl-space-2026/main/data/" + fn, fn)
        with rasterio.open("taipei_s2_sample.tif") as src: cube_s, tags_s = src.read().astype("float32"), src.tags()
        with rasterio.open("taipei_s2_sample_scl.tif") as src: scl_s = src.read(1).astype("uint8")
        np.save("lab0_cube.npy", cube_s); np.save("lab0_scl.npy", scl_s)
        json.dump(dict(scene_id=tags_s.get("scene_id", "course sample"), tile=tags_s.get("tile", "51RTQ"), crs="EPSG:32651", gsd_m=10, course_sample=True), open("lab0_meta.json", "w"))
        USE_SAMPLE = True; print("Using the course's bundled Sentinel-2 sample (real data). Say so in your card.")
    except Exception as e:
        print("sample not available:", repr(e)[:90])
USE_SYNTHETIC = not os.path.exists("lab0_cube.npy")
print("synthetic fallback:", USE_SYNTHETIC)

## 1 · Load the cube and define the label
Label values: **1** water, **0** not water, **-1** ignore (cloud, shadow, nodata). The ignore value is part of the contract; every score below masks it out.

In [ ]:
if not USE_SYNTHETIC:
    cube = np.load("lab0_cube.npy").astype("float32"); scl = np.load("lab0_scl.npy")
    meta = json.load(open("lab0_meta.json"))
else:
    rng = np.random.default_rng(0); H, W = 768, 1024
    yy, xx = np.mgrid[0:H, 0:W] / 100.0
    river = np.abs(yy - 3.5 - 1.3*np.sin(xx/1.6)) < 0.25; sea = xx < 1.3 + 0.3*np.sin(yy); lake = ((xx-7)**2 + (yy-5.5)**2) < 0.5
    water = river | sea | lake; urban = (xx > 4) & (yy > 4.5) & ~water; veg = ~water & ~urban
    def band(wv, vv, uv, n=0.02):
        a = np.where(water, wv, np.where(urban, uv, vv)).astype("float32"); return np.clip(a + rng.normal(0, n, a.shape), 0.001, 0.95)
    cube = np.stack([band(.06,.04,.12), band(.08,.07,.14), band(.05,.05,.16), band(.02,.45,.22), band(.01,.25,.30), band(.005,.15,.28)])
    cloud = ((xx-8.5)**2 + (yy-1.5)**2) < 0.7; cube[:, cloud] = 0.7
    scl = np.where(water, 6, np.where(urban, 5, 4)).astype("uint8"); scl[cloud] = 9; cube[:, :20, :] = np.nan; scl[:20, :] = 0
    meta = dict(scene_id="SYNTHETIC", tile="TEST", crs="EPSG:32651", gsd_m=10)
BANDS = ["blue", "green", "red", "nir", "swir16", "swir22"]; I = {b: i for i, b in enumerate(BANDS)}
valid = np.isin(scl, [4, 5, 6, 7, 11]) & np.isfinite(cube).all(0)
label = np.where(valid, (scl == 6).astype("int8"), -1)
print("cube", cube.shape, "| valid %.1f %%" % (100*valid.mean()), "| water share of valid %.1f %%" % (100*(label==1).sum()/valid.sum()))

### 1b · Histograms per band per class — the feasibility check
If water and not-water overlap in every band, no threshold and no small model will separate them.

In [ ]:
fig, axes = plt.subplots(1, 6, figsize=(18, 2.8))
for ax, b in zip(axes, BANDS):
    v = cube[I[b]]
    ax.hist(v[label == 0], bins=80, alpha=.6, color="#657681", label="not water", density=True)
    ax.hist(v[label == 1], bins=80, alpha=.7, color="#3066BE", label="water", density=True)
    ax.set_title(b); ax.set_yticks([])
axes[0].legend(frameon=False, fontsize=8); plt.tight_layout(); plt.show()

## 2 · Chip the cube
Every chip keeps its `row`, `col`, `tile`, `date`, and `valid` fraction. Without the first two you cannot split honestly later.

In [ ]:
CH = 128; MIN_VALID = 0.7
rows = range(0, cube.shape[1] - CH + 1, CH); cols = range(0, cube.shape[2] - CH + 1, CH)
records = []
for r in rows:
    for c in cols:
        v = valid[r:r+CH, c:c+CH].mean()
        if v < MIN_VALID: continue
        wf = (label[r:r+CH, c:c+CH] == 1).sum() / max(1, valid[r:r+CH, c:c+CH].sum())
        records.append(dict(row=r, col=c, tile=meta.get("tile", "?"), date=meta.get("scene_id", "?")[-30:], valid=round(float(v), 3), water_frac=round(float(wf), 4)))
print(len(records), "chips of", len(rows)*len(cols), "kept (valid >= %.0f %%)" % (100*MIN_VALID))
print("chips with any water:", sum(r["water_frac"] > 0.01 for r in records))

## 3 · Two splits of the same chips
Split A assigns chips at random. Split B groups chips into 2 × 2 blocks (2.56 km squares; use 4 × 4 on a full tile) and assigns whole blocks, with a one-chip buffer around the test blocks dropped from training.

In [ ]:
rng = np.random.default_rng(2026)
for rec in records: rec["split_A"] = str(rng.choice(["train", "val", "test"], p=[.6, .2, .2]))
BLK = 2                                              # 2 x 2 chips = 2.56 km blocks (use 4 for a full tile)
blocks = sorted({(rec["row"] // (BLK*CH), rec["col"] // (BLK*CH)) for rec in records}); rng.shuffle(blocks)
nb_ = len(blocks); n_test = max(1, round(.2*nb_)); n_val = max(1, round(.2*nb_))
bmap = {b: ("test" if i < n_test else "val" if i < n_test + n_val else "train") for i, b in enumerate(blocks)}
test_blocks = {b for b, s in bmap.items() if s == "test"}
def near_test(b): return any(max(abs(b[0]-t[0]), abs(b[1]-t[1])) == 1 for t in test_blocks)
for rec in records:
    b = (rec["row"] // (BLK*CH), rec["col"] // (BLK*CH)); s = bmap[b]
    rec["split_B"] = "buffer" if (s == "train" and near_test(b)) else s     # buffer chips are excluded from training
if sum(r["split_B"] == "train" for r in records) < 0.3 * len(records):       # tiny scene: buffer would eat the train set
    print("scene too small for a buffer - keeping buffer chips in train (say so on the card)")
    for rec in records:
        if rec["split_B"] == "buffer": rec["split_B"] = "train"
for k in ["split_A", "split_B"]:
    cnt = collections.Counter(r[k] for r in records)
    wf = {s: np.mean([r["water_frac"] for r in records if r[k] == s]) for s in cnt}
    print(k, dict(cnt), "| mean water fraction per split:", {s: round(v, 3) for s, v in wf.items()})

In [ ]:
# visualise the two splits on the scene grid
colors = {"train": "#CFE3D8", "val": "#D6E0F5", "test": "#F6D9C9", "buffer": "#EEEEEE"}
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, k in zip(axes, ["split_A", "split_B"]):
    g = np.full((len(rows), len(cols), 3), 1.0)
    for rec in records:
        i, j = rec["row"] // CH, rec["col"] // CH
        g[i, j] = [int(colors[rec[k]][m:m+2], 16)/255 for m in (1, 3, 5)]
    ax.imshow(g, interpolation="nearest"); ax.set_title(k + ("  (random chips)" if k == "split_A" else "  (spatial blocks + buffer)")); ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()

## 4 · Baseline 1 — Otsu on NDWI
The threshold is fitted on **train** chips only, then applied to every split. Scores are per split, per class (water), and ignore `-1` pixels.

In [ ]:
ndwi = (cube[I["green"]] - cube[I["nir"]]) / (cube[I["green"]] + cube[I["nir"]] + 1e-6)
def mask_of(split_key, part):
    m = np.zeros(valid.shape, bool)
    for rec in records:
        if rec[split_key] == part: m[rec["row"]:rec["row"]+CH, rec["col"]:rec["col"]+CH] = True
    return m & valid
def scores(pred, part_mask):
    p, y = pred[part_mask].astype(bool), (label[part_mask] == 1)
    tp, fp, fn, tn = (p&y).sum(), (p&~y).sum(), (~p&y).sum(), (~p&~y).sum()
    return dict(precision=tp/max(1,tp+fp), recall=tp/max(1,tp+fn), IoU=tp/max(1,tp+fp+fn), accuracy=(tp+tn)/max(1,tp+fp+fn+tn), n=int(p.size))
results = []
for k in ["split_A", "split_B"]:
    T = threshold_otsu(ndwi[mask_of(k, "train")])
    pred = ndwi > T
    for part in ["train", "val", "test"]:
        s = scores(pred, mask_of(k, part)); s.update(method="Otsu-NDWI", split=k, part=part, T=round(float(T), 3)); results.append(s)
import pandas as pd
df = pd.DataFrame(results); print(df[["method","split","part","T","precision","recall","IoU","accuracy"]].round(3).to_string(index=False))

## 5 · Baseline 2 — random forest on bands + indices + texture
Eight features per pixel: six bands, NDVI, NDWI, plus a 5 × 5 local standard deviation of NIR (texture). Trained on a sample of train pixels, scored per split.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
ndvi = (cube[I["nir"]] - cube[I["red"]]) / (cube[I["nir"]] + cube[I["red"]] + 1e-6)
nir = np.nan_to_num(cube[I["nir"]]); tex = np.sqrt(np.maximum(ndimage.uniform_filter(nir**2, 5) - ndimage.uniform_filter(nir, 5)**2, 0))
feats = np.concatenate([np.nan_to_num(cube), ndvi[None], ndwi[None], tex[None]], 0)          # (9, H, W)
X_all = feats.reshape(feats.shape[0], -1).T
for k in ["split_A", "split_B"]:
    tr = mask_of(k, "train").ravel(); idx = np.flatnonzero(tr)
    idx = rng.choice(idx, size=min(60000, idx.size), replace=False)                            # subsample for speed
    rf = RandomForestClassifier(n_estimators=100, max_depth=12, class_weight="balanced", n_jobs=-1, random_state=0)
    rf.fit(X_all[idx], (label.ravel()[idx] == 1))
    pred = rf.predict(X_all).reshape(valid.shape)
    for part in ["train", "val", "test"]:
        s = scores(pred, mask_of(k, part)); s.update(method="RandomForest", split=k, part=part, T=None); results.append(s)
df = pd.DataFrame(results)
table = df.pivot_table(index=["method", "split"], columns="part", values="IoU").round(3)[["train", "val", "test"]]
print("water IoU by method x split x part\n", table)
gap = {m: round(float(df[(df.method==m)&(df.split=="split_A")&(df.part=="test")].IoU.iloc[0] - df[(df.method==m)&(df.split=="split_B")&(df.part=="test")].IoU.iloc[0]), 3) for m in ["Otsu-NDWI", "RandomForest"]}
print("\nleakage gap (test IoU random split - block split):", gap)

## 6 · From pixels to objects
Clean the NDWI mask with morphology, label connected components, drop blobs under 20 pixels (0.2 ha at 10 m), count what remains.

In [ ]:
T = threshold_otsu(ndwi[mask_of("split_B", "train")])
m = (ndwi > T) & valid
m = ndimage.binary_opening(m, iterations=1); m = ndimage.binary_closing(m, iterations=1)
lab = measure.label(m, connectivity=2); props = [p for p in measure.regionprops(lab) if p.area >= 20]
print(len(props), "water bodies >= 20 px; largest = %d px = %.1f ha" % (max(p.area for p in props), max(p.area for p in props)*100/1e4))
fig, axes = plt.subplots(1, 2, figsize=(12, 5)); axes[0].imshow(np.where(valid, ndwi, np.nan), cmap="Blues"); axes[0].set_title("NDWI")
axes[1].imshow(np.isin(lab, [p.label for p in props]), cmap="gray"); axes[1].set_title(f"{len(props)} water objects after cleaning")
for ax in axes: ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()

## 7 · Hand-check ten chips
Pick ten chips that contain water (or should). For each, look at the true-colour chip and the SCL water label side by side and record whether the **label** is right. This is your label-noise estimate for section 3 of the card.

In [ ]:
def stretch(a):
    lo, hi = np.nanpercentile(a, [2, 98]); return np.clip((a - lo) / (hi - lo + 1e-9), 0, 1)
cands = sorted([r for r in records if r["water_frac"] > 0.02], key=lambda r: -r["water_frac"])[:10]
fig, axes = plt.subplots(2, 10, figsize=(20, 4.4))
for j, rec in enumerate(cands):
    r, c = rec["row"], rec["col"]; rgb = np.stack([stretch(cube[I[b]][r:r+CH, c:c+CH]) for b in ["red", "green", "blue"]], -1)
    axes[0, j].imshow(np.nan_to_num(rgb)); axes[0, j].set_title(f"chip {j+1}  r{r} c{c}", fontsize=8)
    axes[1, j].imshow(label[r:r+CH, c:c+CH], cmap="Blues", vmin=-1, vmax=1)
    for ax in axes[:, j]: ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()

✏️ **YOUR HAND-CHECK TABLE** (edit this cell)

| chip | row | col | label looks right? (yes / partly / no) | what is wrong |
|---|---|---|---|---|
| 1 | | | | |
| 2 | | | | |
| 3 | | | | |
| 4 | | | | |
| 5 | | | | |
| 6 | | | | |
| 7 | | | | |
| 8 | | | | |
| 9 | | | | |
| 10 | | | | |

Agreement estimate (yes / 10): ___

## 8 · Save the dataset

In [ ]:
X = np.stack([np.nan_to_num(cube[:, r["row"]:r["row"]+CH, r["col"]:r["col"]+CH]) for r in records]).astype("float32")
Y = np.stack([label[r["row"]:r["row"]+CH, r["col"]:r["col"]+CH] for r in records]).astype("int8")
np.savez_compressed("chips.npz", X=X, Y=Y)
pd.DataFrame(records).to_csv("chips.csv", index=False)
df.round(4).to_csv("baseline_results.csv", index=False)
card_numbers = dict(n_chips=len(records), chip=CH, gsd_m=meta.get("gsd_m", 10), bands=BANDS, ignore_value=-1,
                    splits={k: dict(collections.Counter(r[k] for r in records)) for k in ["split_A", "split_B"]},
                    water_share_valid=round(float((label==1).sum()/valid.sum()), 4), leakage_gap_iou=gap,
                    otsu_T_splitB=round(float(T), 3), scene=meta.get("scene_id"), synthetic=USE_SYNTHETIC)
json.dump(card_numbers, open("card_numbers.json", "w"), indent=2); print(json.dumps(card_numbers, indent=2))
print("\nSaved chips.npz (X", X.shape, ", Y", Y.shape, "), chips.csv, baseline_results.csv, card_numbers.json")

## 9 · ✏️ YOUR ANSWERS (marked)
1. **Split.** Report test IoU for Otsu and for the random forest under split A and split B. Explain the gap in two sentences using the word *autocorrelation*.
2. **Metric.** Your accuracy numbers are all high. Why are they useless here, and which metric would you put on the dataset card as primary? Which failure (FP or FN) is unacceptable for your seed problem?
3. **Labels.** From the hand-check: what does SCL water get wrong in your window? Would you use it as a training label? As a test label?
4. **Baseline.** The random forest uses one texture feature. Which feature family from the lecture would you add next for your seed problem, and why?
5. **Card.** Paste the seven section headings of the card and one line each for your team's seed problem (this is the draft you finish in the template).

✏️ **YOUR ANSWER**

1. 

2. 

3. 

4. 

5. 

---
### Checklist before you export
- [ ] All cells executed; not the synthetic fallback
- [ ] Split table, IoU table and leakage gap printed
- [ ] Hand-check table filled (10 rows)
- [ ] `chips.npz`, `chips.csv`, `card_numbers.json` produced; the card (template) filled and uploaded with this PDF

*Contains modified Copernicus Sentinel data 2026.*